In [1]:
try:
    import FinanceDataReader
except:
    %pip install finance-datareader

In [2]:
# 초급용 캐시 사용 코드
import FinanceDataReader as fdr
import pandas as pd

from datetime import datetime, timedelta
from pathlib import Path
from tqdm import tqdm


# ==================================================
# 1. 기본 설정
# ==================================================

# 오늘 날짜에서 600일 전 날짜를 구한다.
# 예: 오늘이 2026-08-17이면 약 600일 전부터 주가를 받는다.
START = (datetime.now() - timedelta(days=600)).strftime("%Y-%m-%d")

# 주가 데이터를 저장할 폴더
CACHE = Path("stock_data")

# stock_cache 폴더가 없으면 새로 만든다.
CACHE.mkdir(exist_ok=True)


# ==================================================
# 2. KOSPI 종목 목록
# ==================================================

print("KOSPI 종목 목록 확인 중...")

# 종목 목록을 저장할 파일
list_file = CACHE / "KOSPI_list.csv"


try:
    # 인터넷에서 최신 KOSPI 종목 목록을 가져온다.
    stocks = fdr.StockListing("KOSPI")

except Exception as e:
    # 인터넷 오류 등으로 다운로드가 실패하면
    # 이전에 저장해 둔 종목 목록을 사용한다.
    print("종목 목록 다운로드 실패 → 기존 파일 사용")
    print(e)

    # 기존 파일마저 없으면 프로그램을 중단한다.
    if not list_file.exists():
        raise RuntimeError("저장된 KOSPI 종목 목록도 없습니다.")

    stocks = pd.read_csv(
        list_file,
        dtype={"Code": str}
    )


# 종목코드를 문자열로 바꾸고 6자리로 맞춘다.
# 예: 5930 → "005930"
stocks["Code"] = (
    stocks["Code"]
    .astype(str)
    .str.zfill(6)
)


# 종목코드 마지막 숫자가 0인 일반 종목만 선택한다.
stocks = stocks[
    stocks["Code"].str.endswith("0")
].copy()


# 다음 실행에서도 사용할 수 있도록 종목 목록을 저장한다.
stocks.to_csv(
    list_file,
    index=False
)

print("분석 종목 :", len(stocks))


# ==================================================
# 3. KOSPI 지수 데이터
# ==================================================

print()
print("KOSPI 지수 확인 중...")


# KOSPI 지수를 저장할 파일
kospi_file = CACHE / "KS11.csv"


# --------------------------------------------------
# KOSPI 파일이 없는 경우
# --------------------------------------------------

if not kospi_file.exists():

    # 최근 600일 KOSPI 지수를 처음부터 다운로드한다.
    kospi = fdr.DataReader(
        "KS11",
        START
    )

    # 데이터가 없으면 프로그램을 중단한다.
    if kospi.empty:
        raise RuntimeError(
            "KOSPI 데이터를 가져오지 못했습니다."
        )

    # 날짜를 날짜 형식으로 변경한다.
    kospi.index = pd.to_datetime(kospi.index)

    # 오래된 날짜 → 최신 날짜 순으로 정렬한다.
    kospi = kospi.sort_index()

    # CSV 파일로 저장한다.
    kospi.to_csv(
        kospi_file,
        index_label="Date"
    )


# --------------------------------------------------
# KOSPI 파일이 이미 있는 경우
# --------------------------------------------------

else:

    # 저장해 둔 KOSPI 파일을 읽는다.
    kospi = pd.read_csv(
        kospi_file,
        index_col="Date",
        parse_dates=["Date"]
    )

    # 날짜순으로 정렬한다.
    kospi = kospi.sort_index()


    # CSV 파일은 있지만 데이터가 없는 경우
    if kospi.empty:

        # 최근 600일 데이터를 다시 다운로드한다.
        kospi = fdr.DataReader(
            "KS11",
            START
        )

        if kospi.empty:
            raise RuntimeError(
                "KOSPI 데이터를 가져오지 못했습니다."
            )

        kospi.index = pd.to_datetime(kospi.index)
        kospi = kospi.sort_index()


    # 기존 KOSPI 데이터가 있는 경우
    else:

        try:
            # 마지막 저장 날짜보다 5일 앞부터 다시 다운로드한다.
            #
            # 예:
            # 마지막 날짜가 8월 10일이면
            # 약 8월 5일부터 다시 받는다.
            start = (
                kospi.index[-1]
                - pd.Timedelta(days=5)
            ).strftime("%Y-%m-%d")


            # 최근 KOSPI 데이터를 가져온다.
            new = fdr.DataReader(
                "KS11",
                start
            )


            # 새 데이터가 있는 경우
            if not new.empty:

                # 날짜 형식으로 변경하고 정렬한다.
                new.index = pd.to_datetime(new.index)
                new = new.sort_index()

                # 기존 데이터와 새 데이터를 합친다.
                kospi = pd.concat([
                    kospi,
                    new
                ])

                # 같은 날짜가 두 번 있으면
                # 새로 다운로드한 데이터를 남긴다.
                kospi = kospi[
                    ~kospi.index.duplicated(
                        keep="last"
                    )
                ].sort_index()


        except Exception as e:
            # 최신 데이터 다운로드가 실패하더라도
            # 기존 데이터를 계속 사용한다.
            print(
                "KOSPI 갱신 실패 → 기존 데이터 사용"
            )
            print(e)


    # --------------------------------------------------
    # 최근 600일만 남기기
    # --------------------------------------------------

    # 600일보다 오래된 데이터는 삭제한다.
    cutoff = (
        pd.Timestamp.today()
        - pd.Timedelta(days=600)
    )

    kospi = kospi[
        kospi.index >= cutoff
    ]


    # 최신 상태의 KOSPI 데이터를 다시 저장한다.
    kospi.to_csv(
        kospi_file,
        index_label="Date"
    )


# KOSPI 데이터가 비어 있는지 마지막으로 확인한다.
if kospi.empty:
    raise RuntimeError(
        "KOSPI 데이터가 없습니다."
    )


# KOSPI 데이터의 마지막 날짜를
# 전체 시장의 최신 거래일로 사용한다.
market_date = kospi.index[-1].date()

print("최신 거래일 :", market_date)


# ==================================================
# 4. 결과를 저장할 변수
# ==================================================

# 처음 다운로드한 종목 수
download_count = 0

# 기존 CSV에 최신 데이터를 추가한 종목 수
update_count = 0

# 이미 최신이라 다운로드하지 않은 종목 수
skip_count = 0

# 새로 받을 데이터가 없는 종목 수
no_new_count = 0

# 오류가 발생한 종목 수
error_count = 0


# ==================================================
# 5. 종목별 주가 데이터
# ==================================================

# tqdm을 사용해서 진행률을 표시한다.
pbar = tqdm(
    stocks.iterrows(),
    total=len(stocks),
    desc="주가 확인"
)


# KOSPI 종목을 하나씩 처리한다.
for _, stock in pbar:

    # 현재 종목의 종목코드
    code = stock["Code"]

    # 현재 종목의 회사 이름
    name = stock["Name"]

    # 이 종목의 주가를 저장할 CSV 파일
    # 예: stock_cache/005930.csv
    stock_file = CACHE / f"{code}.csv"


    try:

        # --------------------------------------------------
        # ① CSV 파일이 없으면 처음부터 다운로드
        # --------------------------------------------------

        if not stock_file.exists():

            # 최근 600일 주가를 다운로드한다.
            df = fdr.DataReader(
                code,
                START
            )

            # 데이터가 없으면 다음 종목으로 넘어간다.
            if df.empty:
                no_new_count += 1
                continue

            # 날짜를 날짜 형식으로 변경한다.
            df.index = pd.to_datetime(df.index)

            # 날짜순으로 정렬한다.
            df = df.sort_index()

            # CSV 파일로 저장한다.
            df.to_csv(
                stock_file,
                index_label="Date"
            )

            download_count += 1

            # 이 종목 처리가 끝났으므로
            # 다음 종목으로 넘어간다.
            continue


        # --------------------------------------------------
        # ② CSV 파일이 있으면 기존 데이터를 읽는다.
        # --------------------------------------------------

        df = pd.read_csv(
            stock_file,
            index_col="Date",
            parse_dates=["Date"]
        )

        # 날짜순으로 정렬한다.
        df = df.sort_index()


        # --------------------------------------------------
        # ③ CSV가 비어 있으면 다시 전체 다운로드
        # --------------------------------------------------

        if df.empty:

            df = fdr.DataReader(
                code,
                START
            )

            if df.empty:
                no_new_count += 1
                continue

            df.index = pd.to_datetime(df.index)
            df = df.sort_index()

            df.to_csv(
                stock_file,
                index_label="Date"
            )

            download_count += 1

            continue


        # --------------------------------------------------
        # ④ 마지막 저장 날짜 확인
        # --------------------------------------------------

        # 기존 CSV의 가장 최근 날짜
        last_date = df.index[-1].date()


        # KOSPI 최신 거래일까지 데이터가 있다면
        # 추가 다운로드할 필요가 없다.
        if last_date >= market_date:

            skip_count += 1

            continue


        # --------------------------------------------------
        # ⑤ 부족한 최근 데이터만 다운로드
        # --------------------------------------------------

        # 마지막 날짜보다 5일 앞부터 다시 다운로드한다.
        #
        # 예:
        # 기존 마지막 날짜 : 8월 10일
        # 다시 받는 날짜   : 약 8월 5일부터
        start = (
            df.index[-1]
            - pd.Timedelta(days=5)
        ).strftime("%Y-%m-%d")


        # 최근 데이터만 다운로드한다.
        new = fdr.DataReader(
            code,
            start
        )


        # 새 데이터가 없으면 다음 종목으로 넘어간다.
        if new.empty:

            no_new_count += 1

            continue


        # 새 데이터의 날짜를 날짜 형식으로 바꾼다.
        new.index = pd.to_datetime(new.index)

        # 날짜순으로 정렬한다.
        new = new.sort_index()


        # --------------------------------------------------
        # ⑥ 기존 데이터 + 새 데이터 합치기
        # --------------------------------------------------

        df = pd.concat([
            df,
            new
        ])


        # 5일 정도 겹쳐서 받았기 때문에
        # 같은 날짜가 두 번 있을 수 있다.
        #
        # 중복 날짜가 있으면 새 데이터를 사용한다.
        df = df[
            ~df.index.duplicated(
                keep="last"
            )
        ].sort_index()


        # --------------------------------------------------
        # ⑦ 최근 600일 데이터만 남기기
        # --------------------------------------------------

        cutoff = (
            pd.Timestamp.today()
            - pd.Timedelta(days=600)
        )

        df = df[
            df.index >= cutoff
        ]


        # 갱신된 데이터를 CSV에 다시 저장한다.
        df.to_csv(
            stock_file,
            index_label="Date"
        )

        update_count += 1


    # --------------------------------------------------
    # 특정 종목에서 오류가 발생한 경우
    # --------------------------------------------------

    except Exception as e:

        error_count += 1

        # 오류가 발생해도 프로그램을 중단하지 않고
        # 다음 종목을 계속 처리한다.
        tqdm.write(
            f"오류 : {name} → {e}"
        )


# ==================================================
# 6. 결과
# ==================================================

print()
print("=" * 40)
print("데이터 준비 완료")
print("=" * 40)

print("최신 거래일    :", market_date)
print("전체 분석 종목 :", len(stocks))

print()

print("신규 다운로드  :", download_count)
print("기존 파일 갱신 :", update_count)
print("이미 최신      :", skip_count)
print("새 데이터 없음 :", no_new_count)
print("오류           :", error_count)

print()

print("저장 위치      :", CACHE)

KOSPI 종목 목록 확인 중...
분석 종목 : 833

KOSPI 지수 확인 중...
최신 거래일 : 2026-08-28


주가 확인: 100%|██████████████████████████████| 833/833 [01:39<00:00,  8.41it/s]


데이터 준비 완료
최신 거래일    : 2026-08-28
전체 분석 종목 : 833

신규 다운로드  : 0
기존 파일 갱신 : 833
이미 최신      : 0
새 데이터 없음 : 0
오류           : 0

저장 위치      : stock_data


In [3]:
# end